In [ ]:
!pip install squarify


In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
sys.path.append('..')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import mannwhitneyu, ttest_ind

import utilities.functions as functions

from utilities.graficos import plot_metricas
from utilities.graficos import (
  boxplot_meses
 
)


from utilities.functions import (
    gerar_stats,
    pedidos_group,
    criacao_ordens,
    conversao_imediata,
   
)

from utilities.testes_estatisticos import testes,teste_proporcao_por_janela
from utilities.outliers import(
    outlier_method,
    mark_outliers_iqr_zscore_mad
)

In [2]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 
df = pd.read_parquet(BASE_PATH / "gold" / "df_publico.parquet")

In [ ]:
df=df[df['order_created_month']==12]#.head(1000)
df.head()

,customer_id,is_target,active,created_at,delivery_address_state,merchant_id,order_created_at,order_id,order_total_amount,origin_platform,order_created_month,unique_order_hash,weekday,hour,day,total_amount_mes,ticket_medio,num_pedidos_mes,num_pedidos_hist,outlier_iqr,outlier_zscore,outlier_mad,lim_inf_iqr,lim_sup_iqr,zscore,mad_score,id_p99,id_p10
17,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True,2018-04-06T02:48:42.887Z,SP,468f55386b6b45c114f292ba03ae6250012ef2aeda3cea...,2018-12-05 14:10:06+00:00,c773ed2e871b02f60a1812163b2953adb040b05e753808...,9.50,ANDROID,12,17691586763885382211,Wednesday,14,5,22.50,11.25,2,19,False,False,False,0.01,102.15,-1.06,-1.39,0,1
18,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,True,2018-04-06T02:48:42.887Z,SP,05291eacc3cb88847b56af15838a0e653a1fcda54a1d88...,2018-12-04 14:16:44+00:00,6a91dd04ba929af216ec1a8144e3992d552b281916cb9b...,13.00,ANDROID,12,3535338040383315140,Tuesday,14,4,22.50,11.25,2,19,False,False,False,0.01,102.15,-0.96,-1.23,0,1
24,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,True,2018-04-06T03:54:45.752Z,RJ,65b0d6b2fb8f46a7f5ff34e92a027ba16daede54050e8a...,2018-12-31 23:28:52+00:00,a182cc69e4a36e675eab25579595d51a48056aa304af78...,20.00,ANDROID,12,5222660554591170869,Monday,23,31,20.00,20.00,1,6,False,False,False,0.01,102.15,-0.14,-0.91,0,0
45,d425d6ee4c9d4e211b71da8fc60bf6c5336b2ea9af9cc0...,control,True,2018-01-06T21:04:49.159Z,SP,2f1f388732fe98108ab117e89954ebf8fcb8abee50e116...,2018-12-12 22:15:22+00:00,a31ea26614781a1efca03a9cda135f4abf79d7c6e5b2e9...,45.00,IOS,12,16377365159234429210,Wednesday,22,12,703.70,63.97,11,31,False,False,False,0.01,102.15,-0.01,0.24,0,0
46,d425d6ee4c9d4e211b71da8fc60bf6c5336b2ea9af9cc0...,control,True,2018-01-06T21:04:49.159Z,SP,12e7a61b43666ef30af2440923b2945f6e694a3b85d27c...,2018-12-10 21:24:56+00:00,9341f4389b614cb02194d624312ec385cdde0fcd0634b0...,70.00,DESKTOP,12,3044180806641378589,Monday,21,10,703.70,63.97,11,31,False,False,False,0.01,102.15,0.11,1.39,0,0


In [4]:
outlier_method(df,var='order_total_amount')

Use MAD (assimétrico + robusto)


/opt/anaconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 1234460.
  res = hypotest_fun_out(*samples, **kwds)


'mad'

In [7]:
df = mark_outliers_iqr_zscore_mad(df)


In [9]:
df_outliers = df[
    df['outlier_iqr'] & 
    df['outlier_zscore'] & 
    df['outlier_mad']]
print(len(df_outliers))
df_outliers.describe()

11172


,order_total_amount,order_created_month,unique_order_hash,hour,day,total_amount_mes,ticket_medio,num_pedidos_mes,num_pedidos_hist,lim_inf_iqr,lim_sup_iqr,zscore,mad_score,id_p99,id_p10
count,"11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00","11,172.00"
mean,232.89,12.00,"9,180,313,424,420,005,888.00",16.91,17.96,652.82,177.31,4.53,12.27,0.01,102.15,4.76,8.86,0.76,0.04
std,"1,320.01",0.00,"5,348,061,397,350,541,312.00",7.39,8.17,"1,966.97",946.52,4.43,12.39,0.00,0.00,7.32,60.57,0.43,0.20
min,155.75,12.00,"1,273,739,674,839,548.00",0.00,3.00,155.80,35.91,1.00,2.00,0.01,102.15,3.00,5.32,0.00,0.00
25%,169.00,12.00,"4,489,944,665,927,910,400.00",15.00,11.00,262.50,110.20,2.00,4.00,0.01,102.15,3.37,5.93,1.00,0.00
50%,189.80,12.00,"9,217,359,302,302,629,888.00",21.00,17.00,432.00,149.62,3.00,8.00,0.01,102.15,3.94,6.88,1.00,0.00
75%,229.00,12.00,"13,792,256,801,776,175,104.00",22.00,25.00,756.02,188.53,6.00,16.00,0.01,102.15,5.02,8.68,1.00,0.00
max,"138,750.90",12.00,"18,446,323,495,608,420,352.00",23.00,31.00,"140,338.90","70,169.45",82.00,239.00,0.01,102.15,704.08,"6,364.67",1.00,1.00


In [11]:
df=df[~df['customer_id'].isin(df_outliers['customer_id'].unique())]

In [13]:
data_maxima = df['order_created_at'].max()


In [ ]:
#df=df[df['customer_id']!='361e229dbc1b985e1aacb3e70384782a05d77ad6db53e7e511fe2147ee09a890']

In [ ]:
rfm = df.groupby(['customer_id','is_target']).agg({
    'order_created_at': lambda x: (data_maxima - x.max()).days,  
    'unique_order_hash': 'count',  
    'order_total_amount': 'sum'  
}).reset_index()

In [ ]:
rfm.head()

In [ ]:

rfm.columns = ['customer_id','is_target', 'recencia', 'frequencia', 'valor_monetario']

rfm.head()

In [ ]:
rfm.shape

In [ ]:

for col, inv in [('recencia', True), ('frequencia', False), ('valor_monetario', False)]:
    rank = rfm[col].rank(method='first')
    if inv:
        rfm[f'{col[0]}_score'] = pd.qcut(rank, q=4, labels=[4,3,2,1])
    else:
        rfm[f'{col[0]}_score'] = pd.qcut(rank, q=4, labels=[1,2,3,4])

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['v_score'].astype(str)



In [ ]:
rfm.reset_index().sort_values(by=['f_score'], ascending=[False]).head()


In [ ]:
df_stats_mes = (
        rfm.groupby(['rfm_score','is_target'])
          .agg(
              total_clientes=('customer_id', 'nunique'),
              media=("valor_monetario", 'mean'))) .reset_index().sort_values(by=['media'], ascending=[False])

df_stats_mes.head()

In [ ]:
rfm_avg = rfm.pivot_table(
    index='r_score', 
    columns='f_score', 
    values='valor_monetario', 
    aggfunc='mean'
)


In [ ]:
# Média do Valor Monetário por célula RFM

plt.figure(figsize=(10, 6))
sns.heatmap(rfm_avg, annot=True, fmt='.0f', cmap='YlOrRd', cbar_kws={'label': 'Valor Médio (R$)'})
plt.title('Valor Médio por Segmento RFM')
plt.xlabel('F Score (Frequência)')
plt.ylabel('R Score (Recência)')
plt.show()


In [ ]:

rfm_treemap = rfm_avg.reset_index().melt(
    id_vars='r_score',
    var_name='f_score',
    value_name='valor_medio'
)

In [ ]:
rfm_treemap = rfm_avg.reset_index().melt(
    id_vars='r_score',
    var_name='f_score',
    value_name='valor_medio'
)

# 2) Ordenar por R e depois por F
rfm_treemap = rfm_treemap.sort_values(['r_score', 'f_score']).reset_index(drop=True)

# 3) Criar rótulos
rfm_treemap["label"] = (
    "R:" + rfm_treemap["r_score"].astype(str) +
    " | F:" + rfm_treemap["f_score"].astype(str) +
    "\nR$ " + rfm_treemap["valor_medio"].round(0).astype(str)
)

# 4) Normalizar valores para o colormap
norm_vals = rfm_treemap["valor_medio"] / rfm_treemap["valor_medio"].max()
colors = plt.cm.RdYlGn(norm_vals)   # verde = alto valor, vermelho = baixo


# 5) Plot do Treemap
plt.figure(figsize=(12, 7))

squarify.plot(
    sizes=rfm_treemap["valor_medio"],
    label=rfm_treemap["label"],
    color=colors,
    alpha=0.9,
    text_kwargs={'fontsize': 10, 'color': 'black'}
)

plt.title("Treemap RFM — Valor Médio por Segmento (R x F)", fontsize=15)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cria uma tabela de mapeamento R+F -> segmento
seg_map = pd.DataFrame({
    'r_score': [4, 3, 2, 1, 4, 4, 3, 3,3,4,1,2,2,1,1,2],
    'f_score': [4, 4, 4, 4, 1, 2, 1, 2,3,3,3,3,2,1,2,1],
    'segmento': [
        "media 281",
        "media 220",
        "media 182",
        "media 158",
        "media 47-50",
        "media 47-50",
        "media 47-50",
        "media 47-50",
        "media 94-96",
        "media 94-96",
        "media 78",
        "media 89",
        "media 47-50",
        "media 47-50",
        "media 47-50",
        "media 47-50"
    ]
})


rfm_segmentado = rfm.merge(seg_map, on=['r_score', 'f_score'], how='left')#.reset_index().sort_values(by=['valor_monetario'], ascending=[False])



In [ ]:
rfm_segmentado.head()

In [ ]:
boxplot_meses(rfm_segmentado,var_cat='segmento',var_cont='valor_monetario')